In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
  repo_path = Path("/content/dolphin")
  if not (repo_path / ".git").exists():
    subprocess.run(
      ["git", "clone", "--depth", "1", "https://github.com/angelraymondi/dolphin.git", str(repo_path)],
      check=True,
    )
  os.chdir(repo_path)

%pip install -r requirements.txt

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm
from util.engines.servi import search, get_raw_url
from util.fetch_video import fetch_video
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  from google.colab import drive as google_drive
  google_drive.mount("/content/drive")
  OUTPUT_DIR = Path("/content/drive/MyDrive/Dolphin")
else:
  OUTPUT_DIR = Path("downloads")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find(query):
  results = search(query, cyclic=True)
  return results.get("video_urls", [])


def secure(video_urls, max_workers=10):
  raw_urls = [None] * len(video_urls)

  with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
      executor.submit(get_raw_url, url): i
      for i, url in enumerate(video_urls)
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Resolving URLs", unit="video"):
      idx = futures[future]
      try:
        raw_urls[idx] = future.result()
      except Exception as e:
        tqdm.write(f"Error resolving {video_urls[idx]}: {e}")

  return [url for url in raw_urls if url]


def download(video_urls, max_workers=5, output_dir=None):
  output_path = Path(output_dir) if output_dir else OUTPUT_DIR
  output_path.mkdir(parents=True, exist_ok=True)

  successful = []
  failed = []

  with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
      executor.submit(fetch_video, url, str(output_path / f"{i}.mp4")): url
      for i, url in enumerate(video_urls, start=1)
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading", unit="video"):
      url = futures[future]
      try:
        future.result()
        successful.append(url)
      except Exception as e:
        failed.append((url, str(e)))
        tqdm.write(f"Error downloading {url}: {e}")

  return successful, failed

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

query_input = widgets.Text(
  placeholder="Enter your video search query...",
  description="Search:",
  layout=widgets.Layout(width="75%")
)

output_dir_input = widgets.Text(
  value=str(OUTPUT_DIR),
  description="Output:",
  layout=widgets.Layout(width="75%")
)

workers_input = widgets.IntSlider(
  value=5,
  min=1,
  max=20,
  description="Workers:",
  continuous_update=False
)

start_button = widgets.Button(
  description="Start Download",
  button_style="success",
  icon="download",
  layout=widgets.Layout(width="200px")
)

status = widgets.HTML(value="<b>Status:</b> Waiting for input")

search_progress = widgets.IntProgress(
  value=0,
  min=0,
  max=1,
  description="Search:",
  bar_style="info",
  layout=widgets.Layout(width="100%")
)

resolve_progress = widgets.IntProgress(
  value=0,
  min=0,
  max=1,
  description="Resolve:",
  bar_style="info",
  layout=widgets.Layout(width="100%")
)

download_progress = widgets.IntProgress(
  value=0,
  min=0,
  max=1,
  description="Download:",
  bar_style="info",
  layout=widgets.Layout(width="100%")
)

log_output = widgets.Output(
  layout=widgets.Layout(
    border="1px solid #888",
    padding="10px",
    width="100%",
    max_height="300px",
    overflow="auto"
  )
)


def run_download(_):
  query = query_input.value.strip()
  output_dir = output_dir_input.value.strip()

  if not query:
    status.value = "<b style='color:red'>Please enter a search query.</b>"
    return

  if not output_dir:
    status.value = "<b style='color:red'>Please enter an output directory.</b>"
    return

  start_button.disabled = True

  search_progress.value = 0
  resolve_progress.value = 0
  download_progress.value = 0

  for progress in [search_progress, resolve_progress, download_progress]:
    progress.max = 1
    progress.bar_style = "info"

  with log_output:
    clear_output(wait=True)

  def log(message):
    with log_output:
      print(message)

  try:
    status.value = "<b>Current process:</b> Searching"

    video_urls = find(query)
    search_progress.value = 1

    log(f"Found {len(video_urls)} videos.")

    if not video_urls:
      status.value = "<b style='color:orange'>No videos found.</b>"
      return

    status.value = "<b>Current process:</b> Resolving URLs"

    raw_urls = [None] * len(video_urls)

    resolve_progress.max = len(video_urls)

    with ThreadPoolExecutor(max_workers=10) as executor:
      futures = {
        executor.submit(get_raw_url, url): i
        for i, url in enumerate(video_urls)
      }

      for future in as_completed(futures):
        idx = futures[future]

        try:
          raw_urls[idx] = future.result()
        except Exception as e:
          log(f"Resolution error: {video_urls[idx]} - {e}")

        resolve_progress.value += 1

    safe_urls = [url for url in raw_urls if url]

    resolve_progress.bar_style = "success"
    log(f"Resolved {len(safe_urls)}/{len(video_urls)} URLs.")

    if not safe_urls:
      status.value = "<b style='color:orange'>No valid URLs found.</b>"
      return

    status.value = "<b>Current process:</b> Downloading"

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    download_progress.max = len(safe_urls)

    successful = 0
    failed = 0

    with ThreadPoolExecutor(max_workers=workers_input.value) as executor:
      futures = {
        executor.submit(
          fetch_video,
          url,
          str(Path(output_dir) / f"{i}.mp4")
        ): url
        for i, url in enumerate(safe_urls, start=1)
      }

      for future in as_completed(futures):
        try:
          future.result()
          successful += 1
        except Exception as e:
          failed += 1
          log(f"Download error: {futures[future]} - {e}")

        download_progress.value += 1

    download_progress.bar_style = "success" if failed == 0 else "warning"

    log(f"\nSuccessful downloads: {successful}")
    log(f"Failed downloads: {failed}")
    log(f"Output directory: {Path(output_dir).resolve()}")

    status.value = (
      f"<b style='color:green'>Completed!</b> "
      f"{successful}/{len(safe_urls)} videos downloaded."
    )

  except Exception as e:
    status.value = f"<b style='color:red'>Error:</b> {e}"
    log(f"Pipeline error: {e}")

  finally:
    start_button.disabled = False


start_button.on_click(run_download)

interface = widgets.VBox([
  widgets.HTML("<h3>Video Downloader</h3>"),
  query_input,
  output_dir_input,
  workers_input,
  start_button,
  widgets.HTML("<hr>"),
  status,
  search_progress,
  resolve_progress,
  download_progress,
  widgets.HTML("<h4>Process Log</h4>"),
  log_output
])

display(interface)